In [ ]:
!cutadapt -a GGCAAATGGGTTTTCAAGATCC...GGTANTGGANNTATGAATATTC --revcomp --overlap 18 -e 0.2 --discard-untrimmed --minimum-length 200 --maximum-length 450 --info-file data/cox2_th_cutadapt/cutadapt-info.tsv -o data/cox2_th_cutadapt/seq-trimmed_tmp.fasta data/cox2_th_cutadapt/seq-filtered.fasta

This is cutadapt 5.1 with Python 3.10.18
Command line parameters: -a GGCAAATGGGTTTTCAAGATCC...GGTANTGGANNTATGAATATTC --overlap 18 --revcomp -e 0.2 --discard-untrimmed --minimum-length 200 -o data/cut-test-require-both.fasta data/cox2_th/seq-filtered.fasta
Processing single-end reads on 1 core ...
Done           00:00:00         3,873 reads @ 198.6 µs/read;   0.30 M reads/minute

=== Summary ===

Total reads processed:                   3,873
Reads with adapters:                     3,859 (99.6%)
Reverse-complemented:                        0 (0.0%)

== Read fate breakdown ==
Reads that were too short:                 169 (4.4%)
Reads discarded as untrimmed:               14 (0.4%)
Reads written (passing filters):         3,690 (95.3%)

Total basepairs processed:     5,748,580 bp
Total written (filtered):      1,027,740 bp (17.9%)

=== Adapter 1 ===

Sequence: GGCAAATGGGTTTTCAAGATCC...GGTANTGGANNTATGAATATTC; Type: linked; Length: 22+22; 5' trimmed: 218 times; 3' trimmed: 3858 times; Rev

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('data/cox2_th_cutadapt/cutadapt-info.tsv', sep='\t', header=None)
df = df.drop(columns=[8, 9, 10])
df.columns = ['accession', 'errors', 'start', 'end', 'left_seq', 'primer_seq', 'right_seq', 'primer_name', 'reverse_complement']
df = df[df['errors'] != -1]
df['accession'] = df['accession'].str.replace(' rc', '')
df['reverse_complement'] = df['reverse_complement'].astype(bool)
df['start'] = df['start'].astype(int)
df['end'] = df['end'].astype(int)
df['primer_size'] = df['end'] - df['start']
df

,accession,errors,start,end,left_seq,primer_seq,right_seq,primer_name,reverse_complement,primer_size
0,PV995066.1,0,242,264,GTTTTTTTTAATTATTGTTACTGTATTTGTATGTTGGATGTTATTT...,GGTATTGGAGTTATGAATATTC,AGATAATTTAGAATTTTCTGATGAACCTTTAATTTTTGATAGTTAT...,1;2,False,22
1,PV995065.1,0,242,264,GTTTTTTTTAATTATTGTTACTGTATTTGTATGTTGGATGTTATTT...,GGTATTGGAGTTATGAATATTC,AGATAATTTAGAATTTTCTGATGAACCTTTAATTTTTGATAGTTAT...,1;2,False,22
2,PV995064.1,0,242,264,GTTTTTTTTAATTATTGTTACTGTATTTGTATGTTGGATGTTATTT...,GGTATTGGAGTTATGAATATTC,AGATAATTTAGAATTTTCTGATGAACCTTTAATTTTTGATAGTTAT...,1;2,False,22
3,PV956201.1,0,247,269,TTAATGTTTTTTTTAATTATCGTTACTGTATTTGTTTGTTGGATGT...,GGTACTGGAGTTATGAATATTC,CGATAATTTAGAATTTTCCGATGAACCTTTAATTTTCGATAGTTAT...,1;2,False,22
4,PV956200.1,0,290,312,AGCAACTCCAGTTATGGAAGGTATTATTAACTTTCATCATGATTTA...,GGTACTGGAGTTATGAATATTC,CGATAATTTAGAATTTTCCGATGAACCTTTAATTTTCGATAGTTAT...,1;2,False,22
...,...,...,...,...,...,...,...,...,...,...
4708,AB253782.1,1,284,306,TGCGACTCCTGTAATGGAAGGTATTATTAATTTTCATAATGACTTA...,GGTATTGGAGTTATGAATATAC,TGATAATTTTAATAATATTAATTTTGATAGTTATATGATTCCTACT...,1;2,False,22
4709,AB253781.1,1,284,306,TGCGACTCCTGTAATGGAAGGTATTATTAATTTTCATAATGACTTA...,GGTATTGGAGTTATGAATATAC,TGATAATTTTAATAATATTAATTTTGATAGTTATATGATTCCTACT...,1;2,False,22
4710,AB253780.1,1,284,306,TGCGACTCCTGTAATGGAAGGTATTATTAATTTTCATAATGACTTA...,GGTATTGGAGTTATGAATATAC,TGATAATTTTAATAATATTAATTTTGATAGCTATATGATTCCTACT...,1;2,False,22
4711,AY913800.1,1,291,313,TGCAACTCCACTGTATGGAAGGTATTATAAATTTTCATCATGATTT...,GGTTTTGGACTTATGAATATTC,AGATATTTTAAATTACTTTGATAAAAATGAAAATTTAATTTTTGAT...,1;2,False,22


In [3]:
# Example primer mapping (adjust if you have more than one pair)
primer_map = {"1;1": "forward", "1;2": "reverse"}

# Assign forward/reverse labels
df["primer_direction"] = df["primer_name"].map(primer_map)

# Pivot so forward/reverse get their own columns
wide = df.pivot_table(
    index="accession",
    columns="primer_direction",
    values=["errors", "start", "end", "left_seq", "primer_seq", "right_seq", "primer_name", 'primer_size'],
    aggfunc="first"  # avoids duplicates
)

# Flatten the MultiIndex column names
wide.columns = [f"{col[0]}_{col[1]}" for col in wide.columns]

# Reset index so accession is a normal column
wide = wide.reset_index()
wide = wide.drop_duplicates(subset="accession")

# Keep accession first, then all forward, then all reverse
ordered_cols = ["accession", "start_forward", "end_forward", "errors_forward", 'primer_size_forward', "primer_name_forward", "primer_seq_forward", "left_seq_forward", "right_seq_forward", "start_reverse", "end_reverse", 'primer_size_reverse', "errors_reverse", "primer_name_reverse", "primer_seq_reverse", "left_seq_reverse", "right_seq_reverse"]
wide = wide[ordered_cols]
wide = wide.merge(df[['accession', 'reverse_complement']], on='accession', how='inner')
wide = wide.drop_duplicates(subset="accession")

wide

,accession,start_forward,end_forward,errors_forward,primer_size_forward,primer_name_forward,primer_seq_forward,left_seq_forward,right_seq_forward,start_reverse,end_reverse,primer_size_reverse,errors_reverse,primer_name_reverse,primer_seq_reverse,left_seq_reverse,right_seq_reverse,reverse_complement
0,AB095053.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,287.0,309.0,22.0,0.0,1;2,GGTATTGGAGTTATGAATATTC,AACTCCAGTAATGGAAGGTATTATTAACTTTCATCATGATTTAATG...,CGATAATTTAGAATTTGCTGATGAACCTTTAATTTTTGATAGTTAT...,False
1,AB095054.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,287.0,309.0,22.0,0.0,1;2,GGTATTGGAGTTATGAATATTC,AACTCCAGTAATGGAAGGTATTATTAACTTTCATCATGATTTAATG...,CGATAATTTGGAATTTGCTGACGAACCTTTAATTTTTGATAGTTAT...,False
2,AB095055.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,287.0,309.0,22.0,0.0,1;2,GGTATTGGAGTTATGAATATTC,AACTCCAGTAATGGAAGGTATTATTAACTTTCATCATGATTTAATG...,CGATAATTTGGAATTTGCTGACGAACCTTTAATTTTTGATAGTTAT...,False
3,AB095056.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,287.0,309.0,22.0,0.0,1;2,GGTATTGGAGTTATGAATATTC,AACTCCAGTAATGGAAGGTATTATTAACTTTCATCATGATTTAATG...,CGATAATTTAGAATTTGCTGATGAACCTTTAATTTTTGATAGTTAT...,False
4,AB095057.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,287.0,309.0,22.0,1.0,1;2,GGTATTGGAGTTATGAGTATTC,AACACCTGTAATGGAAGGTATTATAAACTTTCATCATGATTTAATG...,TGATAATTTAGAATTCGCAGATGAACCTTTAATTTTTGATAGTTAT...,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4075,PV956200.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,290.0,312.0,22.0,0.0,1;2,GGTACTGGAGTTATGAATATTC,AGCAACTCCAGTTATGGAAGGTATTATTAACTTTCATCATGATTTA...,CGATAATTTAGAATTTTCCGATGAACCTTTAATTTTCGATAGTTAT...,False
4076,PV956201.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,247.0,269.0,22.0,0.0,1;2,GGTACTGGAGTTATGAATATTC,TTAATGTTTTTTTTAATTATCGTTACTGTATTTGTTTGTTGGATGT...,CGATAATTTAGAATTTTCCGATGAACCTTTAATTTTCGATAGTTAT...,False
4077,PV995064.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,242.0,264.0,22.0,0.0,1;2,GGTATTGGAGTTATGAATATTC,GTTTTTTTTAATTATTGTTACTGTATTTGTATGTTGGATGTTATTT...,AGATAATTTAGAATTTTCTGATGAACCTTTAATTTTTGATAGTTAT...,False
4078,PV995065.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,242.0,264.0,22.0,0.0,1;2,GGTATTGGAGTTATGAATATTC,GTTTTTTTTAATTATTGTTACTGTATTTGTATGTTGGATGTTATTT...,AGATAATTTAGAATTTTCTGATGAACCTTTAATTTTTGATAGTTAT...,False


In [4]:
"""wide_copy = wide.copy()
wide_copy['start_reverse'] = wide_copy['start_reverse'].fillna(0)
wide_copy['end_forward'] = wide_copy['end_forward'].fillna(0)
wide_copy['amplicon_size'] = wide_copy['start_reverse'] - wide_copy['end_forward']
amplicon_size = wide_copy[['accession', 'amplicon_size']]
wide = wide.merge(amplicon_size, on='accession')
wide"""

"wide_copy = wide.copy()\nwide_copy['start_reverse'] = wide_copy['start_reverse'].fillna(0)\nwide_copy['end_forward'] = wide_copy['end_forward'].fillna(0)\nwide_copy['amplicon_size'] = wide_copy['start_reverse'] - wide_copy['end_forward']\namplicon_size = wide_copy[['accession', 'amplicon_size']]\nwide = wide.merge(amplicon_size, on='accession')\nwide"

In [5]:
both_primers = wide.dropna(subset=['primer_seq_forward', 'primer_seq_reverse']).copy()
# fix reverse primer position
both_primers['start_reverse'] = both_primers['end_forward'] + both_primers['start_reverse']
both_primers['end_reverse'] = both_primers['end_forward'] + both_primers['end_reverse']
both_primers

,accession,start_forward,end_forward,errors_forward,primer_size_forward,primer_name_forward,primer_seq_forward,left_seq_forward,right_seq_forward,start_reverse,end_reverse,primer_size_reverse,errors_reverse,primer_name_reverse,primer_seq_reverse,left_seq_reverse,right_seq_reverse,reverse_complement
416,DQ365699.1,2.0,23.0,1.0,21.0,1;1,GGCAAATGGGTTTCAAGATCC,TT,AGCTACACCTGTTATGGAAGGAATTATTAATTTTCACCACGATATT...,313.0,335.0,22.0,0.0,1;2,GGTACTGGAGTTATGAATATTC,AGCTACACCTGTTATGGAAGGAATTATTAATTTTCACCACGATATT...,AGATAATTTAGAATTTTCTGATGAACCTTTAATTTTTGATAGTTAT...,False
418,DQ365700.1,1.0,23.0,0.0,22.0,1;1,GGCAAATGGGTTTTCAAGATCC,T,AGCAACTCCAGTTATGGAAGGTATTATTAACTTTCATCATGATATA...,313.0,335.0,22.0,0.0,1;2,GGTATTGGAGTTATGAATATTC,AGCAACTCCAGTTATGGAAGGTATTATTAACTTTCATCATGATATA...,AGATAATTTAGAATTTTCAGATGAACCTTTAATTTTTGATAGTTAT...,False
420,DQ365702.1,1.0,22.0,1.0,21.0,1;1,GGCAAATGGGTTTCAAGATCC,T,TGCAACTCCAGTAATGGAAGGTATTATTAACTTTCATCATGATTTA...,312.0,334.0,22.0,0.0,1;2,GGTATTGGAGTTATGAATATTC,TGCAACTCCAGTAATGGAAGGTATTATTAACTTTCATCATGATTTA...,AGACAATTTAGAATTTTCAGATGAACCTTTAATTTTTGATAGTTAT...,False
424,DQ365705.1,1.0,22.0,1.0,21.0,1;1,GGCAAATGGGTTTCAAGATCC,T,AGCAACTCCAGTAATGGAAGGTATTATAAACTTTCATCATGATTTA...,312.0,334.0,22.0,0.0,1;2,GGTATTGGAGTTATGAATATTC,AGCAACTCCAGTAATGGAAGGTATTATAAACTTTCATCATGATTTA...,AGATAATTTAGAATTTTCAGATGAACCTTTAATTTTTGATAGTTAT...,False
426,DQ365706.1,0.0,22.0,0.0,22.0,1;1,GGCAAATGGGTTTTCAAGATCC,None,TGCAACTCCCGTTATGGAAGGTATTATTAACTTTCATCACGATTTA...,312.0,334.0,22.0,0.0,1;2,GGTATTGGAGTTATGAATATTC,TGCAACTCCCGTTATGGAAGGTATTATTAACTTTCATCACGATTTA...,AGATAATTTAGAATTTTCGGATGAACCTTTAATTTTTGATAGTTAT...,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4030,PV243791.1,36613.0,36635.0,2.0,22.0,1;1,GGCAATTAGGTTTTCAAGATCC,ATGAATTTTCAAAATATAAATAAATGGTCAACAAGATGGCTTTTTT...,AGCAACTCCCGTAATGGAAGGTATTATAAATTTTCATCATGATTTA...,36925.0,36947.0,22.0,0.0,1;2,GGTACTGGAGTTATGAATATTC,AGCAACTCCCGTAATGGAAGGTATTATAAATTTTCATCATGATTTA...,AGATAATTTAGAATTTTCTGATGAACCTTTAATTTTTGATAGTTAT...,False
4032,PV243795.1,37377.0,37399.0,2.0,22.0,1;1,GGCAATTAGGTTTTCAAGATCC,ATGAATTTTCAAAATATAAATAAATGGTCAACAAGATGGCTTTTTT...,AGCAACTCCAGTAATGGAAGGTATTATAAATTTTCATCATGATTTA...,37689.0,37711.0,22.0,0.0,1;2,GGTACTGGAGTTATGAATATTC,AGCAACTCCAGTAATGGAAGGTATTATAAATTTTCATCATGATTTA...,AGATAATTTAGAATTTTCTGATGAACCTTTAATTTTTGATAGTTAT...,False
4034,PV243796.1,37377.0,37399.0,2.0,22.0,1;1,GGCAATTAGGTTTTCAAGATCC,ATGAATTTTCAAAATATAAATAAATGGTCAACAAGATGGCTTTTTT...,AGCAACTCCAGTAATGGAAGGTATTATAAATTTTCATCATGATTTA...,37689.0,37711.0,22.0,0.0,1;2,GGTACTGGAGTTATGAATATTC,AGCAACTCCAGTAATGGAAGGTATTATAAATTTTCATCATGATTTA...,AGATAATTTAGAATTTTCTGATGAACCTTTAATTTTTGATAGTTAT...,False
4036,PV243803.1,39017.0,39039.0,2.0,22.0,1;1,GGCAATTAGGTTTTCAAGATCC,ATGAATTTTCAAAATATAAATAAATGGTCAACAAGATGGCTTTTTT...,AGCAACTCCAGTTATGGAAGGTATTATAAACTTTCATCATGATTTA...,39329.0,39351.0,22.0,0.0,1;2,GGTATTGGAGTTATGAATATTC,AGCAACTCCAGTTATGGAAGGTATTATAAACTTTCATCATGATTTA...,AGATAATTTAGAATTTTCTGATGAACCTTTAATTTTTGATAGTTAT...,False


In [6]:
final_db = pd.read_csv('data/cox2_th_cutadapt/cox2_th_cutadapt-meta.csv', sep='\t')
final_db

,accession,creation_date,title,authors,sequence_length,organism_name,tax_id_x,organism,organelle,mol_type,...,trimed_seq_len,tax_id_y,species,domain,kingdom,phylum,class,order,family,genus
0,MW006635.1,23-NOV-2020,Globisporangium irregulare isolate UM1087 cyto...,"Callaghan,S.E.",621,Globisporangium irregulare,36331,Globisporangium irregulare,mitochondrion,genomic DNA,...,315,36331,Globisporangium irregulare,Eukaryota,Stramenopiles,Oomycota,Peronosporomycetes,Pythiales,Pythiaceae,Globisporangium
1,KM819519.1,27-JAN-2015,Pythiogeton ramosum isolate SC2.7.2 cytochrome...,"Le,D.P., Smith,M.K. and Aitken,E.",614,Pythiogeton ramosum,1217586,Pythiogeton ramosum,mitochondrion,genomic DNA,...,307,1217586,Pythiogeton ramosum,Eukaryota,Stramenopiles,Oomycota,Peronosporomycetes,Pythiales,Pythiogetonaceae,Pythiogeton
2,LC028397.1,01-NOV-2016,Pythium barbulae mitochondrial cox2 gene for c...,"Ueta,S. and Tojo,M.",613,Globisporangium barbulae,1629122,Globisporangium barbulae,mitochondrion,genomic DNA,...,306,1629122,Globisporangium barbulae,Eukaryota,Stramenopiles,Oomycota,Peronosporomycetes,Pythiales,Pythiaceae,Globisporangium
3,KP723171.1,16-SEP-2015,Globisporangium kandovanense voucher CCTU 1813...,"Chenari Bouket,A., Arzanlou,M., Tojo,M. and Ba...",587,Globisporangium kandovanense,1632203,Globisporangium kandovanense,mitochondrion,genomic DNA,...,301,1632203,Globisporangium kandovanense,Eukaryota,Stramenopiles,Oomycota,Peronosporomycetes,Pythiales,Pythiaceae,Globisporangium
4,GU071804.1,25-JUL-2016,Pythium irregulare strain STE-U6769 cytochrome...,"Spies,C.F., Mazzola,M., Botha,W.J., Langenhove...",596,Globisporangium irregulare,36331,Globisporangium irregulare,mitochondrion,genomic DNA,...,299,36331,Globisporangium irregulare,Eukaryota,Stramenopiles,Oomycota,Peronosporomycetes,Pythiales,Pythiaceae,Globisporangium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1281,MK602199.1,29-APR-2020,Globisporangium spinosum isolate C9 cytochrome...,"Ibanez,J.M., Obregon,V.G. and Favaro,M.A.",518,Globisporangium spinosum,2125998,Globisporangium spinosum,mitochondrion,genomic DNA,...,211,2125998,Globisporangium spinosum,Eukaryota,Stramenopiles,Oomycota,Peronosporomycetes,Pythiales,Pythiaceae,Globisporangium
1282,MK602203.1,29-APR-2020,Globisporangium irregulare isolate C13 cytochr...,"Ibanez,J.M., Obregon,V.G. and Favaro,M.A.",518,Globisporangium irregulare,36331,Globisporangium irregulare,mitochondrion,genomic DNA,...,211,36331,Globisporangium irregulare,Eukaryota,Stramenopiles,Oomycota,Peronosporomycetes,Pythiales,Pythiaceae,Globisporangium
1283,MK602204.1,29-APR-2020,Globisporangium irregulare isolate C14 cytochr...,"Ibanez,J.M., Obregon,V.G. and Favaro,M.A.",518,Globisporangium irregulare,36331,Globisporangium irregulare,mitochondrion,genomic DNA,...,211,36331,Globisporangium irregulare,Eukaryota,Stramenopiles,Oomycota,Peronosporomycetes,Pythiales,Pythiaceae,Globisporangium
1284,MK602213.1,29-APR-2020,Globisporangium irregulare isolate C24 cytochr...,"Ibanez,J.M., Obregon,V.G. and Favaro,M.A.",518,Globisporangium irregulare,36331,Globisporangium irregulare,mitochondrion,genomic DNA,...,211,36331,Globisporangium irregulare,Eukaryota,Stramenopiles,Oomycota,Peronosporomycetes,Pythiales,Pythiaceae,Globisporangium


In [ ]:
primers_in_db = both_primers[both_primers['accession'].isin(final_db['accession'])].copy()
#primers_in_db = primers_in_db.set_index('accession')
primers_in_db

,accession,start_forward,end_forward,errors_forward,primer_size_forward,primer_name_forward,primer_seq_forward,left_seq_forward,right_seq_forward,start_reverse,end_reverse,primer_size_reverse,errors_reverse,primer_name_reverse,primer_seq_reverse,left_seq_reverse,right_seq_reverse,reverse_complement
424,DQ365705.1,1.0,22.0,1.0,21.0,1;1,GGCAAATGGGTTTCAAGATCC,T,AGCAACTCCAGTAATGGAAGGTATTATAAACTTTCATCATGATTTA...,312.0,334.0,22.0,0.0,1;2,GGTATTGGAGTTATGAATATTC,AGCAACTCCAGTAATGGAAGGTATTATAAACTTTCATCATGATTTA...,AGATAATTTAGAATTTTCAGATGAACCTTTAATTTTTGATAGTTAT...,False
426,DQ365706.1,0.0,22.0,0.0,22.0,1;1,GGCAAATGGGTTTTCAAGATCC,None,TGCAACTCCCGTTATGGAAGGTATTATTAACTTTCATCACGATTTA...,312.0,334.0,22.0,0.0,1;2,GGTATTGGAGTTATGAATATTC,TGCAACTCCCGTTATGGAAGGTATTATTAACTTTCATCACGATTTA...,AGATAATTTAGAATTTTCGGATGAACCTTTAATTTTTGATAGTTAT...,False
429,DQ365708.1,1.0,22.0,1.0,21.0,1;1,GGCAAATGGGTTTCAAGATCC,T,AGCAACTCCCGTAATGGAAGGTATTATAAACTTTCATCACGATTTA...,312.0,334.0,22.0,0.0,1;2,GGTACTGGAGTTATGAATATTC,AGCAACTCCCGTAATGGAAGGTATTATAAACTTTCATCACGATTTA...,CGATAATTTAGAATTTTCTGATGAACCTTTAATTTTTGATAGTTAT...,False
432,DQ365710.1,1.0,22.0,1.0,21.0,1;1,GGCAAATGGGTTTCAAGATCC,T,AGCAACCCCTGTAATGGAAGGTATTATCAATTTTCATCACGATTTA...,312.0,334.0,22.0,0.0,1;2,GGTATTGGAGTTATGAATATTC,AGCAACCCCTGTAATGGAAGGTATTATCAATTTTCATCACGATTTA...,GGATAATTTAGAATTTTCAGATGAACCTTTAATTTTTGATAGTTAT...,False
435,DQ365712.1,4.0,25.0,1.0,21.0,1;1,GGCAAATGGGTTTCAAGATCC,CCTT,TGCAACACCCGTGATGGAGGGTATTATTAATTTCCACCACGATATT...,315.0,337.0,22.0,1.0,1;2,GGTATTGGAGTTACGAATATTC,TGCAACACCCGTGATGGAGGGTATTATTAATTTCCACCACGATATT...,GGATAATTTAGAATTTTCAGATGAACCTCTGATCTTTGATAGTTAT...,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4024,PV243786.1,37447.0,37469.0,2.0,22.0,1;1,GGCAATTAGGTTTTCAAGATCC,ATGAATTTTCAAAATATAAATAAATGGTCAACAAGATGGCTTTTTT...,AGCAACTCCTGTAATGGAAGGTATTATAAATTTTCATCATGATTTA...,37759.0,37781.0,22.0,0.0,1;2,GGTATTGGAGCTATGAATATTC,AGCAACTCCTGTAATGGAAGGTATTATAAATTTTCATCATGATTTA...,AGATAATTTAGAATTTTCAGATGAACCTTTAATTTTTGATAGTTAT...,False
4030,PV243791.1,36613.0,36635.0,2.0,22.0,1;1,GGCAATTAGGTTTTCAAGATCC,ATGAATTTTCAAAATATAAATAAATGGTCAACAAGATGGCTTTTTT...,AGCAACTCCCGTAATGGAAGGTATTATAAATTTTCATCATGATTTA...,36925.0,36947.0,22.0,0.0,1;2,GGTACTGGAGTTATGAATATTC,AGCAACTCCCGTAATGGAAGGTATTATAAATTTTCATCATGATTTA...,AGATAATTTAGAATTTTCTGATGAACCTTTAATTTTTGATAGTTAT...,False
4034,PV243796.1,37377.0,37399.0,2.0,22.0,1;1,GGCAATTAGGTTTTCAAGATCC,ATGAATTTTCAAAATATAAATAAATGGTCAACAAGATGGCTTTTTT...,AGCAACTCCAGTAATGGAAGGTATTATAAATTTTCATCATGATTTA...,37689.0,37711.0,22.0,0.0,1;2,GGTACTGGAGTTATGAATATTC,AGCAACTCCAGTAATGGAAGGTATTATAAATTTTCATCATGATTTA...,AGATAATTTAGAATTTTCTGATGAACCTTTAATTTTTGATAGTTAT...,False
4036,PV243803.1,39017.0,39039.0,2.0,22.0,1;1,GGCAATTAGGTTTTCAAGATCC,ATGAATTTTCAAAATATAAATAAATGGTCAACAAGATGGCTTTTTT...,AGCAACTCCAGTTATGGAAGGTATTATAAACTTTCATCATGATTTA...,39329.0,39351.0,22.0,0.0,1;2,GGTATTGGAGTTATGAATATTC,AGCAACTCCAGTTATGGAAGGTATTATAAACTTTCATCATGATTTA...,AGATAATTTAGAATTTTCTGATGAACCTTTAATTTTTGATAGTTAT...,False


In [8]:
from Bio import SeqIO

In [9]:
accessions = primers_in_db['accession'].tolist()
records = [record for record in SeqIO.parse('data/cox2_th_cutadapt/seq-filtered.fasta', 'fasta') if record.id in accessions]

primer_size = 22

with open('align-full-amplicon.fasta', 'w') as out:
    for record in records:
        row = primers_in_db.loc[primers_in_db['accession'] == record.id]
        if row['reverse_complement'].bool():
            record.seq = record.seq.reverse_complement()
        start = int(row['start_forward'])
        end = int(row['end_reverse'])
        if start != 0 and int(row['primer_size_forward']) < primer_size:
            start = max(0, int(row['end_forward']) - primer_size)
        if end != len(record.seq) and int(row['primer_size_reverse']) < primer_size:
            end = min(len(record.seq), int(row['start_reverse']) + primer_size)
        out.write(f">REF_{record.id}\n{str(record.seq)[start:end]}\n")


/tmp/ipykernel_3959991/1779096869.py:9: FutureWarning: Series.bool is now deprecated and will be removed in future version of pandas
  if row['reverse_complement'].bool():
/tmp/ipykernel_3959991/1779096869.py:11: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  start = int(row['start_forward'])
/tmp/ipykernel_3959991/1779096869.py:12: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  end = int(row['end_reverse'])
/tmp/ipykernel_3959991/1779096869.py:13: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  if start != 0 and int(row['primer_size_forward']) < primer_size:
/tmp/ipykernel_3959991/1779096869.py:15: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Us

In [ ]:
from Bio import AlignIO

records = AlignIO.read('data/cox2_th_cutadapt/cox2_th_cutadapt-qiime-msa.fa', 'fasta')

firsts = []
lasts = []


for record in list(records):
    if record.id not in accessions:
        continue
    seq = str(record.seq)
    # find first non-gap
    first = next(i for i, base in enumerate(seq) if base != "-")
    # find last non-gap
    last = len(seq) - next(i for i, base in enumerate(reversed(seq)) if base != "-") - 1
    firsts.append(first)
    lasts.append(last)
firsts = pd.Series(firsts)
lasts = pd.Series(lasts)
# trim at most frequent position among the reference sequences
trim_start = int(firsts.value_counts(sort=True, ascending=False).index[0])
trim_end = int(lasts.value_counts(sort=True, ascending=False).index[0])


In [17]:
# slice alignment: [rows, columns]
trimmed = records[:, trim_start:trim_end + 1]
AlignIO.write(trimmed, "trimmed_alignment.fasta", "fasta")

1